# 02. Предобработка текста

Цель: подготовить два варианта текста для downstream-задач:

- **Pipeline A (`text_clean`)**: минимальная очистка (без лемматизации). Используется как вход для sentence-transformers при построении эмбеддингов. Модели типа SBERT обучены на живом тексте и лемматизация ухудшает качество

- **Pipeline B (`text_lemm`)**: очистка + токенизация (razdel) + лемматизация (pymorphy3). Используется для c-TF-IDF внутри BERTopic при построении представлений тем.

Фильтрация: удаляем посты короче 50 символов (1% датасета, выявлено в EDA `01_eda.ipynb`)

In [1]:
import re
import os
import pandas as pd
from joblib import Parallel, delayed
import pymorphy3
import razdel
from tqdm import tqdm

In [2]:
RANDOM_STATE = 42
N_JOBS = 4

In [3]:
raw_df = pd.read_parquet("data/posts.parquet", engine="fastparquet")

In [4]:
MIN_TEXT_LEN = 50
df = raw_df[raw_df["text"].fillna("").str.len() >= MIN_TEXT_LEN].copy()
df = df.reset_index(drop=True)

print(f"До фильтрации: {len(raw_df):,}")
print(f"После фильтрации: {len(df):,}")
print(f"Отфильтровано: {len(raw_df) - len(df):,} ({(len(raw_df) - len(df)) / len(raw_df) * 100:.1f}%)")

До фильтрации: 3,035,065
После фильтрации: 3,003,217
Отфильтровано: 31,848 (1.0%)


## Pipeline A - Очистка текста для эмбеддингов (`text_clean`)

Удаляем шум, который не несёт семантической нагрузки:
- URL
- Эмодзи и спецсимволы
- Markdown-разметку (`**bold**`, `_italic_`)
- Множественные пробелы

Лемматизацию **не применяем**. трансформерные модели работают лучше с естественным текстом

In [5]:
# Паттерны для очистки
_URL_RE    = re.compile(r"https?://\S+|www\.\S+")
_EMOJI_RE  = re.compile(
    "[\U00010000-\U0010ffff"
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\u2600-\u26FF\u2700-\u27BF]+",
    flags=re.UNICODE
)
_MD_RE     = re.compile(r"[*_`#|>\[\]~]")
_SPACES_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = _URL_RE.sub(" ", text)
    text = _EMOJI_RE.sub(" ", text)
    text = _MD_RE.sub(" ", text)
    text = _SPACES_RE.sub(" ", text)
    return text.strip()

# Быстрый тест
examples = [
    "Трамп поблагодарил Иран 🙏 за предупреждение https://t.me/example **важно**",
    "Просто обычный текст новости без мусора",
    "www.rbc.ru/link_to_article #тег _курсив_"
]
for ex in examples:
    print(f"До:    {ex}")
    print(f"После: {clean_text(ex)}")
    print()

До:    Трамп поблагодарил Иран 🙏 за предупреждение https://t.me/example **важно**
После: Трамп поблагодарил Иран за предупреждение важно

До:    Просто обычный текст новости без мусора
После: Просто обычный текст новости без мусора

До:    www.rbc.ru/link_to_article #тег _курсив_
После: тег курсив



## Pipeline B - Лемматизация для c-TF-IDF (`text_lemm`)

Поверх Pipeline A применяем:
1. **razdel** - токенизация с учётом особенностей русского текста
2. **pymorphy3** - лемматизация (приведение к начальной форме)
3. Фильтрация, оставляем только слова длиной ≥ 3 символа, удаляем стоп-слова и числа

razdel корректно разбивает текст на токены, pymorphy3 приводит каждый токен к нормальной форме

In [7]:
STOPWORDS = {
    "и","в","во","не","что","он","на","я","с","со","как","а","то","все","она",
    "так","его","но","да","ты","к","у","же","вы","за","бы","по","только","ее",
    "мне","было","вот","от","меня","еще","нет","о","из","ему","теперь","когда",
    "даже","ну","вдруг","ли","если","уже","или","ни","быть","был","него","до",
    "вас","нибудь","опять","уж","вам","ведь","там","потом","себя","ничего","ей",
    "может","они","тут","где","есть","надо","ней","для","мы","тебя","их","чем",
    "была","сам","чтоб","без","будто","человек","чего","раз","тоже","себе","под",
    "будет","ж","тогда","кто","этот","того","потому","этого","какой","совсем",
    "ним","здесь","этом","один","почти","мой","тем","чтобы","нее","сейчас","были",
    "куда","зачем","всех","никогда","можно","при","наконец","два","об","другой",
    "хоть","после","над","больше","тот","через","эти","нас","про","всего","них",
    "какая","много","разве","три","эту","моя","впрочем","хорошо","свою","этой",
    "перед","иногда","лучше","чуть","том","нельзя","такой","им","более","всегда",
    "конечно","всю","между"
}

def lemmatize_text(text: str, morph: pymorphy3.MorphAnalyzer) -> str:
    """Pipeline B: clean → tokenize (razdel) → lemmatize (pymorphy3)"""
    text = clean_text(text)
    tokens = [t.text for t in razdel.tokenize(text)]
    lemmas = []
    for token in tokens:
        token_lower = token.lower()
        if len(token_lower) < 3:
            continue
        if token_lower in STOPWORDS:
            continue
        if not token_lower.isalpha():
            continue
        lemma = morph.parse(token_lower)[0].normal_form
        lemmas.append(lemma)
    return " ".join(lemmas)

# Тест
morph_test = pymorphy3.MorphAnalyzer()
test_text = "Президент России подписал указ об экономических санкциях против западных стран"
print(f"Оригинал: {test_text}")
print(f"Лемм.:    {lemmatize_text(test_text, morph_test)}")

Оригинал: Президент России подписал указ об экономических санкциях против западных стран
Лемм.:    президент россия подписать указ экономический санкция против западный страна


In [8]:
def process_chunk(texts: list[str]) -> tuple[list[str], list[str]]:
    """Обрабатывает чанк текстов, возвращает (clean, lemm)."""
    morph = pymorphy3.MorphAnalyzer()  # создаём внутри воркера
    clean_results = [clean_text(t) for t in texts]
    lemm_results  = [lemmatize_text(t, morph) for t in texts]
    return clean_results, lemm_results

# Разбиваем на чанки
CHUNK_SIZE = 10_000
texts = df["text"].tolist()
chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, len(texts), CHUNK_SIZE)]

print(f"Всего документов: {len(texts):,}")
print(f"Чанков: {len(chunks)} по ~{CHUNK_SIZE:,} документов")
print(f"Воркеров: {N_JOBS}")
print("Запускаем обработку...")

results = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(process_chunk)(chunk) for chunk in chunks
)

# Собираем результаты
clean_texts = []
lemm_texts  = []
for clean_chunk, lemm_chunk in results:
    clean_texts.extend(clean_chunk)
    lemm_texts.extend(lemm_chunk)

df["text_clean"] = clean_texts
df["text_lemm"]  = lemm_texts

print(f"\nОбработано {len(df):,} документов")

Всего документов: 3,003,217
Чанков: 301 по ~10,000 документов
Воркеров: 4
Запускаем обработку...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  8.8min
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed: 24.7min
[Parallel(n_jobs=4)]: Done 301 out of 301 | elapsed: 37.9min finished



Обработано 3,003,217 документов


In [9]:
print("--- Примеры предобработки ---\n")
sample_indices = df.sample(5, random_state=RANDOM_STATE).index

for idx in sample_indices:
    print(f"[Оригинал]\n{df.loc[idx, 'text'][:200]}")
    print(f"\n[text_clean]\n{df.loc[idx, 'text_clean'][:200]}")
    print(f"\n[text_lemm]\n{df.loc[idx, 'text_lemm'][:200]}")
    print("-" * 60)

--- Примеры предобработки ---

[Оригинал]
🥶Брр… **В выходные в Свердловской области будет снег, гололед и сильный ветер**

По прогнозам синоптиков, выходные будут по-зимнему холодными. 28 октября в области будет -8...-12 градусов ночью, +1...

[text_clean]
Брр… В выходные в Свердловской области будет снег, гололед и сильный ветер По прогнозам синоптиков, выходные будут по-зимнему холодными. 28 октября в области будет -8...-12 градусов ночью, +1...-4 гра

[text_lemm]
брр выходной свердловский область снег гололёд сильный ветер прогноз синоптик выходной быть холодный октябрь область градус ночью градус день ветер метр секунда воскресение юг регион метель снег ветер
------------------------------------------------------------
[Оригинал]
Как изменится ключевая ставка ЦБ на заседании 26 июля, [читайте в материале РИАМО.](https://riamo.ru/articles/aktsenty/prognozy-po-kljuchevoj-stavke-reshitsja-li-tsb-na-zhestkie-shagi-ili-otlozhit-sho

[text_clean]
Как изменится ключевая ставка ЦБ на засед

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df["lemm_len"]  = df["text_lemm"].str.split().str.len()
df["clean_len"] = df["text_clean"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["lemm_len"].clip(upper=300), bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_title("Распределение длины text_lemm (токены)")
axes[0].set_xlabel("Количество токенов")
axes[0].set_ylabel("Количество постов")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

axes[1].hist(df["clean_len"].clip(upper=2000), bins=60, color="#55A868", edgecolor="white")
axes[1].set_title("Распределение длины text_clean (символы)")
axes[1].set_xlabel("Длина (символы)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("figures/04_[preprocessing]_preprocessed_lengths.png", bbox_inches="tight")
plt.show()

print(f"Медиана токенов после лемматизации: {df['lemm_len'].median():.0f}")

empty_lemm = (df["text_lemm"].str.strip() == "").sum()
print(f"Пустых text_lemm после обработки: {empty_lemm:,} ({empty_lemm/len(df)*100:.2f}%)")

In [ ]:
preprocessed_dataset_path = "data/posts_preprocessed.parquet"

output_cols = ["id_post", "channel_name", "post_date", "text_clean", "text_lemm", "views"]
df[output_cols].to_parquet(preprocessed_dataset_path, engine="fastparquet", index=False)

print(f"Сохранено: {preprocessed_dataset_path}")
print(f"Записей: {len(df):,}")
print(f"Колонки: {output_cols}")